In [1]:
import optuna.visualization
import os

os.makedirs("optuna_results/plots", exist_ok=True)

In [2]:
structure_study = optuna.load_study(
    study_name="study_new",
    storage="sqlite:///optuna_results/study_new.db"
)

In [3]:
fig = optuna.visualization.plot_pareto_front(
    structure_study,
    target_names=["FLOPs", "Accuracy"],
)
fig.write_image("optuna_results/plots/pareto_front_8p3f.png")
fig.show()

In [4]:
print(f"Number of trials on the Pareto front: {len(structure_study.best_trials)}")

trial_with_highest_accuracy = max(structure_study.best_trials, key=lambda t: t.values[1])
print("Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy.number}")
print(f"\tparams: {trial_with_highest_accuracy.params}")
print(f"\tvalues: {trial_with_highest_accuracy.values}")

Number of trials on the Pareto front: 19
Trial with highest accuracy: 
	number: 42
	params: {'num_transformers': 4, 'dim_heads': '128_8', 'dropout': 0.05}
	values: [4.842384, 0.6731774193548387]


In [5]:
import pandas as pd

best_trials = structure_study.best_trials

order = [
    "trial_id",
    "flops",
    "acc",
    "num_transformers",
    "embedding_dim",
    "num_heads",
    "dropout"
]

trial_dicts = []
for t in best_trials:
    entry = {
        "trial_id": t.number,
        "flops": t.values[0],
        "acc": t.values[1],
    }
    entry.update(t.params)
    trial_dicts.append(entry)

df_best = pd.DataFrame(trial_dicts)
df_best[["embedding_dim", "num_heads"]] = df_best["dim_heads"].str.split("_", expand=True).astype(int)
df_best = df_best.drop(columns=["dim_heads"])[order]
# Discard duplicates
df_best = df_best.drop_duplicates(
    subset=["flops", "acc", "num_transformers", "embedding_dim", "num_heads", "dropout"]
).reset_index(drop=True)
# Sort
df_best = df_best.sort_values(by="flops", ascending=True).reset_index(drop=True)
df_best

,trial_id,flops,acc,num_transformers,embedding_dim,num_heads,dropout
0,18,0.026168,0.652452,4,8,2,0.00
1,14,0.032648,0.656371,5,8,2,0.00
2,24,0.039128,0.656403,6,8,2,0.00
3,53,0.089200,0.662548,4,16,2,0.00
4,12,0.111376,0.665290,5,16,2,0.00
5,23,0.244640,0.667097,3,32,2,0.00
6,21,0.325856,0.668452,4,32,2,0.00
7,55,0.407072,0.670016,5,32,2,0.00
8,6,0.931654,0.670500,3,64,4,0.00
9,7,1.241544,0.671532,4,64,2,0.00


In [9]:
df_best.to_csv("best_trials.csv", index=False)

In [6]:
optuna.visualization.plot_param_importances(
    structure_study, target=lambda t: t.values[0], target_name="flops"
)

In [7]:
optuna.visualization.plot_param_importances(
    structure_study, target=lambda t: t.values[1], target_name="accuracy"
)